In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# **Dataset**
To test the code, a random dataset has been created. The main purpose of this dataset is to make sure that the code is working currently according to the paper.

In [2]:
N = 1024
X = torch.empty(N, 2).uniform_(-2, 2)
y = torch.sin(X[:, :1]) + X[:, 1:] ** 2          # (N, 1)

# **Model, algo & params**
Here, we will be creating the model, parameters and the corresponding algorithm

## Parameters
The following parameters have been used in the model training

In [3]:
layer_sizes = [2, 32, 32, 1]
L = len(layer_sizes) - 1

In [4]:
batch_size = 32
n_batches = N // 32
n_epochs = 20
T = 20

rho = 1.
eta_h = 0.1
alpha = 0.1
eta_theta = 5e-3

## Model
Defining model weights and parameters

In [5]:
weights = nn.ParameterList([
    nn.Parameter(torch.empty(layer_sizes[i+1], layer_sizes[i]))
    for i in range(L)
])
for W in weights:
    nn.init.kaiming_normal_(W, nonlinearity="relu")

The model has been defined as:
$$
\begin{aligned}
h_0 &\leftarrow x, \\
h_i &\leftarrow \sigma(W_i h_{i-1}),
\qquad \text{for } i = 1, \ldots, L-1
\end{aligned}
$$

In [6]:
def _forward(x):
    h = [x]
    for k in range(L-1):
        h.append(F.relu(F.linear(h[-1], weights[k])))
    return h

def _output(h):
    return F.linear(h[-1], weights[-1])

The augmented lagrangian method has been defined as:
$$
\mathcal{L}_{\rho}(h, \theta, \lambda) = \frac{1}{2}||y-W_{L}h_{L-1}||^2 + \sum_{i=1}^{L-1} \lambda_{i}(h_i - \sigma(W_i h_{i-1})) + \frac{\rho}{2} \sum_{i=1}^{L-1} ||h_i - \sigma(W_i h_{i-1})||^2
$$

In [10]:
def _augmented_lagrangian(h, lda, y_sample):
    loss = F.mse_loss(_output(h), y_sample)
    for l in range(1, L):
        pred = F.relu(F.linear(h[l-1], weights[l-1]))
        err = h[l] - pred
        loss += (lda[l-1]*err).sum(-1).mean() + 0.5 * rho * (err**2).sum(-1).mean()
    return loss

According to PC-ALM algorithm, the following steps have been defined:

**Primal step**
$$
\begin{aligned}
h_i &\leftarrow h_i - \eta_h \nabla_{h_i} \mathcal{L}_{\rho}(h, \theta, \lambda),
\qquad \text{for } i = 1, \ldots, L-1
\end{aligned}
$$

**Dual step**
$$
\begin{aligned}
\lambda_i &\leftarrow \lambda_i + \alpha (h_i - \sigma (W_i h_{i-1})),
\qquad \text{for } i = 1, \ldots, L-1
\end{aligned}
$$

**Learning step**
$$
\begin{aligned}
W_i &\leftarrow W_i - \eta_{\theta} \frac{1}{|B|} \sum_{(x,y)\in B} \nabla_{W_i} \mathcal{L}_{\rho}(h, \theta, \lambda),
\qquad \text{for } i = 1, \ldots, L
\end{aligned}
$$

In [8]:
def _primal_step(h, lda, y_sample):
    h = [h[0].detach()] + [hi.detach().requires_grad_(True) for hi in h[1:]]
    lag_loss = _augmented_lagrangian(h, lda, y_sample)
    gradients = torch.autograd.grad(lag_loss, h[1:])
    return [h[0]] + [h[i]-eta_h*gradients[i-1] for i in range(1, L)]

def _dual_step(h, lda):
    ld_processed = []
    for i in range(1, L):
        pred = F.relu(F.linear(h[i-1], weights[i-1]))
        err = h[i] - pred
        ld_processed.append(lda[i-1]+alpha*err)
    return ld_processed

def _learning_step(h, lda, y):
    h_learn = [h[0].detach()] + [hi.detach().requires_grad_(True) for hi in h[1:]]
    lag_loss = _augmented_lagrangian(h_learn, lda, y)
    gradients = torch.autograd.grad(lag_loss, list(weights))
    with torch.no_grad():
        for w, g in zip(weights, gradients):
            w -= eta_theta * g
    

# **Training loop**

In [9]:
for epch in range(n_epochs):
    for i in range(n_batches):
        x_batch = X[i*batch_size:(i+1)*batch_size]
        y_batch = y[i*batch_size:(i+1)*batch_size]

        # Initialisation step
        with torch.no_grad():
            h = _forward(x_batch)
            lda = [torch.zeros_like(h[n]) for n in range(1, L)]

        for t in range(T):
            h = _primal_step(h, lda, y_batch)
            lda = _dual_step(h, lda)

        h = _primal_step(h, lda, y_batch)
        _learning_step(h, lda, y_batch)

    with torch.no_grad():
        pred = _output(_forward(X))
        print(f"epoch {epch:3d}  MSE {F.mse_loss(pred, y).item():.6f}")

epoch   0  MSE 1.017198
epoch   1  MSE 0.614786
epoch   2  MSE 0.425154
epoch   3  MSE 0.325583
epoch   4  MSE 0.271608
epoch   5  MSE 0.241497
epoch   6  MSE 0.223795
epoch   7  MSE 0.212880
epoch   8  MSE 0.205624
epoch   9  MSE 0.200403
epoch  10  MSE 0.196403
epoch  11  MSE 0.193182
epoch  12  MSE 0.190452
epoch  13  MSE 0.188078
epoch  14  MSE 0.185958
epoch  15  MSE 0.184043
epoch  16  MSE 0.182298
epoch  17  MSE 0.180700
epoch  18  MSE 0.179224
epoch  19  MSE 0.177875
